In [94]:
from langchain_community.llms import Ollama
from langchain_ollama import OllamaLLM
from langchain_openai   import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
import json


from dotenv import load_dotenv

Start(userinput)--> Planner Node --> Retriever Node --> Reasoner Node --> Formatter Node --End(output)

In [95]:
class AgentState(TypedDict):
    query: str
    plan: str
    blog: str
    evaluation: str
    final: str

In [96]:
model = OllamaLLM(model='qwen2.5:3b',temperature=0.2)
# model = ChatOpenAI(model='gpt-4-0613', temperature=0.2)

In [97]:

## planner
def planner(state: AgentState):
    prompt =f"""
    You are an expert Blogger.
    
    Create a structured blog outline.
    Include:
    - Title
    - Sections (with headings)
    - Key Points per section
    
    Query:
    {state['query']}
    
    """
    
    plan = model.invoke(prompt)
    
    print(f'[Blog planner]\n', plan)
    
    return {'plan':plan}
    

In [98]:
## blog generator
def blog_generator(state: AgentState):
    prompt = f"""
    
    Write a high quality blog using this outline.
    
    Requirements:
    - Engaging intro
    - Clear sections
    - Simple Language
    - Practical insights
    
    Outline
    {state['plan']}
    
    """
    
    blog = model.invoke(prompt)
    print("\n[Blog generator]\n", blog[:500])
    return {'blog': blog}
    

In [99]:
## evaluator
def evaluator(state: AgentState):
    prompt =f""" 
    You are an Expert critic, evaluate this blog on :
    1. Clarity (1-10)
    2. Depth (1-10)
    3. Engagement (1-10)
    
    Also suggest improvements.
    
    Return JSON:
    {{
        "clarity":...,
        "depth": ...,
        "engagement": ...,
        "improvements":"..."
        }}
    
    
    blog:
    {state['blog']}"""
    
    evaluation = model.invoke(prompt)
    print("\n[Evaluation]\n", evaluation)
    return {'evaluation': evaluation}

In [100]:
## Formatter
def formatter(state: AgentState):
    prompt =f""" 
    Create final output JSON:
    
    {{
        "blog": "...",
        "evaluation": "..."
        
        
    }}
    
    Blog:
    {state['blog']}
    
    Evaluation:
    {state['evaluation']}
    """
    
    output = model.invoke(prompt)
    
    try: 
        parsed =json.loads(output)
    except:
        parsed = {
            
            "blog": state["blog"],
            "evaluation" : state["evaluation"],
            "note": "Formatting fallback"
        }
        
    print(f"\n[FINAL OUTPUT]\n", parsed)
    return {"final": parsed}

In [101]:
## build langgraph

graph = StateGraph(AgentState)

graph.add_node("planner",planner)
graph.add_node("blog_generator", blog_generator)
graph.add_node("evaluator",evaluator)
graph.add_node("formatter",formatter)

graph.set_entry_point("planner")

# graph.add_edge(START, "planner")
graph.add_edge("planner","blog_generator")
graph.add_edge("blog_generator","evaluator")
graph.add_edge("evaluator","formatter")
graph.add_edge("formatter",END)


workflow = graph.compile()

In [103]:
result = workflow.invoke({"query": "write a blog on Batman vs Goku"})
print(result["final"])

[Blog planner]
 ### Blog Outline: Batman vs Goku

#### Title: Batman vs Goku: A Comic Book Battle of the Titans

#### Sections:

1. **Introduction**
   - Brief overview of Batman and Goku
   - The concept of a comic book battle between these two iconic characters
   - Key points to be discussed in the blog

2. **Character Background**
   - **Batman**: Introduction to the Dark Knight, his origin, and his iconic gadgets and allies.
   - **Goku**: Overview of Goku’s journey from a Saiyan warrior to a Super Saiyan, his powers, and his allies.
   - **Key Points**: Discuss the unique aspects of each character that make them stand out in their respective universes.

3. **Setting the Stage for the Battle**
   - **Batman**: His approach to crime and his methods of fighting crime.
   - **Goku**: His approach to fighting and his philosophy.
   - **Key Points**: Highlight how their different approaches to fighting crime and battle could play out in a comic book setting.

4. **The Battle**
   - **B

In [ ]:
print(result['final']['evaluation'])

```json
{
    "clarity": 7,
    "depth": 6,
    "engagement": 5,
    "improvements":"The blog could benefit from more detailed character backgrounds and a deeper exploration of their strengths and weaknesses. The battle scenarios could be fleshed out more with specific tactics and strategies. Additionally, incorporating more anecdotes or quotes from comic book sources could enhance the engagement and credibility of the content."
}
```

### Explanation of Improvements:
- **Clarity**: The blog is generally clear, but some sections could be more concise and focused. For example, the character backgrounds could be more streamlined to highlight key points without unnecessary details.
- **Depth**: The blog touches on the characters' backgrounds and approaches, but could delve deeper into specific aspects like their gadgets, allies, and transformation levels. This would provide a more comprehensive understanding of their unique strengths and weaknesses.
- **Engagement**: The blog is already e